# NLP — dataset audit and cleaning

Interactive inspection of the implemented AbdomenAtlas audit and cleaning stages. The notebook calls pure loading/cleaning functions and displays results inline; it does not rewrite the raw or processed datasets.

## Imports and configuration

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))
from scripts.nlp.audit_dataset import load_and_validate, ID, TEXT, COUNT, LOCATION, DIAMETER, ATTENUATION
from scripts.nlp.prepare_nlp_dataset import clean_report
RAW_PATH = ROOT / 'data/raw/nlp/AbdomenAtlas3.0MiniWithMeta.csv'
df = load_and_validate(RAW_PATH)
print({'rows': len(df), 'columns': len(df.columns), 'path': str(RAW_PATH)})

## Input data inspection

In [ ]:
overview = pd.DataFrame({
    'value': [len(df), df[ID].nunique(), int(df[ID].duplicated().sum()), int(df[ID].isna().sum()), int(df[TEXT].isna().sum()), int(df[TEXT].astype('string').str.strip().eq('').fillna(False).sum())]
}, index=['rows', 'unique_ids', 'duplicate_ids', 'missing_ids', 'missing_reports', 'empty_reports'])
display(overview)
display(df[[ID, TEXT, COUNT, LOCATION, DIAMETER, ATTENUATION]].head(3))
display(df.isna().sum().sort_values(ascending=False).head(12).rename('missing').to_frame())

## Target consistency and text cleaning

In [ ]:
count = pd.to_numeric(df[COUNT], errors='coerce')
target_checks = {
    'zero_lesion_rows': int(count.eq(0).sum()),
    'positive_lesion_rows': int(count.gt(0).sum()),
    'missing_counts': int(count.isna().sum()),
    'negative_counts': int(count.lt(0).sum()),
    'positive_missing_location': int((count.gt(0) & df[LOCATION].isna()).sum()),
    'positive_missing_diameter': int((count.gt(0) & pd.to_numeric(df[DIAMETER], errors='coerce').isna()).sum()),
    'positive_missing_attenuation': int((count.gt(0) & df[ATTENUATION].isna()).sum()),
}
display(pd.Series(target_checks, name='count').to_frame())
cleaned = df[TEXT].map(clean_report)
cleaning_preview = pd.DataFrame({'original': df[TEXT].head(5), 'cleaned': cleaned.head(5)})
display(cleaning_preview)
print({'usable_reports': int(cleaned.notna().sum()), 'excluded_by_cleaning': int(cleaned.isna().sum()), 'remaining_missing_after_cleaning': int(cleaned.isna().sum())})

## Inline text inspection

In [ ]:
lengths = cleaned.dropna().str.len()
display(pd.Series({'mean_characters': lengths.mean(), 'median_characters': lengths.median(), 'shortest': lengths.min(), 'longest': lengths.max(), 'duplicate_clean_text_rows': int(cleaned.dropna().str.lower().duplicated(keep=False).sum())}, name='text_quality').to_frame())
sample = df.loc[cleaned.notna(), [ID, TEXT]].head(5).copy()
sample['cleaned_text'] = cleaned.loc[sample.index]
display(sample)

## Summary

The implemented audit identifies missingness and target relationships without deleting data. `clean_report` only normalizes whitespace and returns missing values explicitly. Persisting a cleaned dataset remains an explicit pipeline operation, not a side effect of this notebook.

In [ ]:
assert set([ID, TEXT, COUNT, LOCATION, DIAMETER, ATTENUATION]).issubset(df.columns)
assert len(df) == 9262
print('Audit and cleaning inspection completed without writing files.')